# Image Colorization — Pretrained ResNet18 Encoder

До этого мы обучали кастомную U-Net (40 эпох). Результаты показали, что модель предсказывает слишком маленькие значения каналов `ab`, из-за чего изображения получаются достаточно тусклыми

Одна из возможных причин заключается в том, что похожие объекты могут иметь разные цвета. При использовании L1 Loss модели может быть выгоднее предсказывать значения ближе к нейтральным, чтобы уменьшить ошибку

Попробуем заменить encoder нашей U-Net на предобученную **ResNet18 (ImageNet)**. Такая модель уже умеет извлекать полезные признаки из изображений, что может помочь улучшить качество колоризации

Сначала попробуем использовать ResNet18 с замороженными весами (**feature extraction**), а затем дообучить часть encoder (**fine-tuning**) и сравнить результаты

ResNet18 была предобучена на RGB-изображениях, поэтому ожидает на вход 3 канала. У нас же изображение представлено только одним каналом L из цветового пространства LAB

Продублируем канал L три раза, получив изображение с 3 одинаковыми каналами, чтобы не менять архитектуру ResNet18 и сохранить предобученные веса

Таким образом, входной тензор изменится с [B, 1, 128, 128] на [B, 3, 128, 128]. При этом изображение не станет цветным, поскольку все три канала содержат одинаковую информацию о яркости

In [ ]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torch import nn
from image_colorization.unet import UNet
from image_colorization.resnet_unet import ResNetUNet
from image_colorization.dataset import ColorizationDataset
from image_colorization.color_utils import denormalize_lab
from image_colorization.training import training_epoch, validation_epoch, train_model
from image_colorization.dataset import Dataset
from pathlib import Path
from skimage.color import lab2rgb

In [5]:
t = torch.rand(1, 1, 128, 128)
t.shape

torch.Size([1, 1, 128, 128])

In [6]:
LEARNING_RATE = 1e-3
model = ResNetUNet()
optimizer = torch.optim.AdamW( model.parameters(),lr=LEARNING_RATE)
criterion = nn.L1Loss()

In [ ]:

model.train_model(model, 
                  criterion, 
                  optimizer, 
                  train_loader, 
                  val_loader, 
                  num_epochs, device)